# Notebook 4 – Run fDOG-Assembly

This notebook runs `fdog.assembly` to find the ortholog of **GAPDH** in the
*Rattus norvegicus* (rat) genome assembly prepared in Notebook 2.

---

### Prerequisites
| Requirement | Where it was created |
|---|---|
| Core group | Notebook 1 → `data/core_orthologs/GAPDH/` |
| Assembly | Notebook 2 → `data/assembly_dir/RAT@10116@v1/` |
| Reference proteome | Notebook 3 → `data/fdog_reference/` |
| fDOG installed | `python3 -m pip install fdog` |

## 1 – Configuration

In [7]:
import shutil
import subprocess
from pathlib import Path

from Bio import SeqIO

# ── Paths from previous notebooks ────────────────────────────────────────────
DATA_DIR       = Path("data")
CORE_GROUP_DIR = DATA_DIR / "core_orthologs"   # Notebook 1
ASSEMBLY_DIR   = DATA_DIR / "assembly_dir"     # Notebook 2
FDOG_REF_DIR   = DATA_DIR / "fdog_reference"   # Notebook 3

# ── Gene and reference species ────────────────────────────────────────────────
GENE_NAME = "GAPDH"
REF_SPEC  = "HUMAN@9606@OMA2024"   # must match Notebooks 1 and 3

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = DATA_DIR / "fda_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Gene name        : {GENE_NAME}")
print(f"  Reference species: {REF_SPEC}")
print(f"  Core group dir   : {CORE_GROUP_DIR}")
print(f"  Assembly dir     : {ASSEMBLY_DIR}")
print(f"  Reference data   : {FDOG_REF_DIR}")
print(f"  Output           : {OUTPUT_DIR}")

Configuration:
  Gene name        : GAPDH
  Reference species: HUMAN@9606@OMA2024
  Core group dir   : data/core_orthologs
  Assembly dir     : data/assembly_dir
  Reference data   : data/fdog_reference
  Output           : data/fda_output


## 2 – Pre-flight Checks

In [8]:
def check(label, condition):
    status = "OK" if condition else "MISSING"
    print(f"  [{status}] {label}")
    return condition

all_ok = True

# Core group files (Notebook 1)
gene_dir = CORE_GROUP_DIR / GENE_NAME
all_ok &= check("Core group FASTA", (gene_dir / f"{GENE_NAME}.fa").exists())
all_ok &= check("Core group MSA",   (gene_dir / f"{GENE_NAME}.aln").exists())
all_ok &= check("Core group HMM",   (gene_dir / "hmm_dir" / f"{GENE_NAME}.hmm").exists())

# Assembly (Notebook 2)
has_assembly = ASSEMBLY_DIR.exists() and any(ASSEMBLY_DIR.iterdir())
all_ok &= check("Assembly directory", has_assembly)

# Reference proteome (Notebook 3)
all_ok &= check("coreTaxa_dir",   (FDOG_REF_DIR / "coreTaxa_dir").exists())

# refSpec present in core group
core_fasta = gene_dir / f"{GENE_NAME}.fa"
if core_fasta.exists():
    ref_found = any(
        REF_SPEC in line
        for line in open(core_fasta)
        if line.startswith(">")
    )
    all_ok &= check(f"refSpec '{REF_SPEC}' in core group", ref_found)

# External tools
for tool in ["fdog.assembly", "miniprot", "blastp"]:
    all_ok &= check(f"Tool: {tool}", shutil.which(tool) is not None)

print()
if all_ok:
    print("All checks passed. Ready to run fDOG-Assembly.")
else:
    print("Some checks failed – fix the issues above before continuing.")

  [OK] Core group FASTA
  [OK] Core group MSA
  [OK] Core group HMM
  [OK] Assembly directory
  [OK] coreTaxa_dir
  [OK] refSpec 'HUMAN@9606@OMA2024' in core group
  [OK] Tool: fdog.assembly
  [OK] Tool: miniprot
  [OK] Tool: blastp

All checks passed. Ready to run fDOG-Assembly.


## 3 – Run fDOG-Assembly

| Parameter | Value | Explanation |
|---|---|---|
| `--gene` | GAPDH | Core ortholog group name |
| `--refSpec` | HUMAN@9606@OMA2024 | Reference species for backward validation |
| `--coregroupPath` | data/core_orthologs/ | Core group folder (Notebook 1) |
| `--assemblyPath` | data/assembly_dir/ | Target genome assembly (Notebook 2) |
| `--dataPath` | data/fdog_reference/ | Reference proteome folder (Notebook 3) |
| `--out` | data/fda_output/ | Output directory |
| `--fast` | flag | Use miniprot only – no MetaEuk DB needed |
| `--fasOff` | flag | Skips FAS feature-architecture scoring to reduce runtime in this example.  In general using FAS is recommanded. |

In [9]:
cmd = [
    "fdog.assembly",
    "--gene",          GENE_NAME,
    "--refSpec",       REF_SPEC,
    "--coregroupPath", str(CORE_GROUP_DIR),
    "--assemblyPath",  str(ASSEMBLY_DIR),
    "--dataPath",      str(FDOG_REF_DIR),
    "--out",           str(OUTPUT_DIR),
    "--fast",
    "--fasOff",
    "--force",
]

print("Command:")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError(f"fdog.assembly exited with code {result.returncode}.")
else:
    print("\nfDOG-Assembly finished successfully.")

Command:
fdog.assembly --gene GAPDH --refSpec HUMAN@9606@OMA2024 --coregroupPath data/core_orthologs --assemblyPath data/assembly_dir --dataPath data/fdog_reference --out data/fda_output --fast --fasOff --force

Gene: GAPDH
fDOG reference species: HUMAN@9606@OMA2024 

Building a consensus sequence
	 ...finished



  0%|          | 0/1 [00:00<?, ?it/s]

################################
Pipeline completed successfully in 12.34s
  Group preparation: 0.02s
  Ortholog search: 12.2s
  FAS calculation: 0s
Output directory:
  /home/hannah/Dev/fdog-assembly/fDOG-Assembly/examples/data/fda_output/GAPDH/
Generated files:
  - GAPDH_og.fa
  - GAPDH.phyloprofile

fDOG-Assembly finished successfully.


100%|██████████| 1/1 [00:12<00:00, 12.18s/it]


## 4 – Inspect the Output

| File | Contents |
|---|---|
| `GAPDH_og.fa` | Reference sequences + predicted ortholog(s) from the assembly |
| `GAPDH.phyloprofile` | Presence/absence table for use with PhyloProfile |

Sequence IDs in `GAPDH_og.fa` follow the format:
`GAPDH|SPECIES@TAXID@VERSION|GENE_ID|1` (best ortholog) or `|0` (co-ortholog).

## Summary – Full Workflow

| Notebook | Task | Key output |
|---|---|---|
| 1 | Core ortholog group from OMA | `data/core_orthologs/GAPDH/` |
| 2 | Genome assembly from NCBI | `data/assembly_dir/RAT@10116@v1/` |
| 3 | Reference proteome from UniProt | `data/fdog_reference/coreTaxa_dir/HUMAN@9606@OMA2024/` |
| 4 | Run fDOG-Assembly | `data/fda_output/GAPDH/GAPDH_og.fa` |